# Cac thuat toan toi uu hoa trong tf.keras: GD, Momentum, Adam

**Muc tieu bai hoc:**
- Hieu **mini-batch gradient descent** - vi sao chia du lieu thanh cac lo (batch) nho thay vi dung
  toan bo du lieu (hoac tung diem mot) cho moi lan cap nhat trong so.
- Hieu ban chat cua 3 thuat toan cap nhat trong so pho bien nhat: **Gradient Descent (GD)**,
  **Momentum**, va **Adam** - va vi sao Adam thuong hoi tu nhanh hon trong thuc te.
- Biet chon va cau hinh optimizer trong `tf.keras` bang `keras.optimizers`.

In [ ]:
# --- Google Colab setup: bo qua tu dong neu chay Jupyter local ---
import sys, os

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # tat toi uu oneDNN de dam bao ket qua giong het nhau
# giua cac lan chay - phai dat TRUOC khi 'import tensorflow' o cell sau (oneDNN co the doi thu
# tu tinh toan tren nhieu luong CPU, gay lech ket qua nho o vai chu so thap phan cuoi giua cac lan)

if "google.colab" in sys.modules:
    REPO_DIR = "/content/AI-Teaching-Labs"
    if not os.path.isdir(REPO_DIR):
        !git clone --depth 1 https://github.com/NonomiyaIzumi/AI-Teaching-Labs.git {REPO_DIR}
    NOTEBOOK_DIR = f"{REPO_DIR}/Deep Learning/Module 1/keras-tensorflow/04_SGD-Momentum-Adam-Toi-uu-hoa"
    os.chdir(NOTEBOOK_DIR)
    print("Da chuyen working directory sang:", os.getcwd())
else:
    print("Khong phai Colab - dung working directory hien tai:", os.getcwd())
    print("Neu chay Jupyter local: tu thu muc keras-tensorflow/, chay 'uv sync' truoc.")

## 1. Mini-batch gradient descent

Voi tap du lieu co `m` vi du, co 3 cach chia du lieu cho moi lan cap nhat trong so:

- **Batch (full-batch) gradient descent:** dung **toan bo** `m` vi du cho moi lan cap nhat. On
  dinh nhung cham voi du lieu lon (phai duyet het du lieu moi khi cap nhat mot lan).
- **Stochastic gradient descent (SGD thuan):** dung **tung 1** vi du cho moi lan cap nhat. Nhanh
  nhung rat "on ao" (nhieu), gradient dao dong manh giua cac buoc.
- **Mini-batch gradient descent:** chia du lieu thanh cac **lo nho** (vd 32, 64, 128 vi du/lo),
  cap nhat mot lan sau moi lo. Dung hoa giua toc do (nhu SGD) va do on dinh (nhu full-batch) - day
  la lua chon pho bien nhat trong thuc te.

Trong `tf.keras`, `model.fit(X, y, batch_size=64, epochs=...)` tu dong: (1) **xao tron (shuffle)**
du lieu train truoc moi epoch, (2) chia thanh cac mini-batch kich thuoc 64, (3) chay forward/
backward/cap nhat cho tung batch. Mot **epoch** la mot lan di het toan bo du lieu (nhieu lan cap
nhat, moi lan tren mot mini-batch).

## 2. Ba thuat toan cap nhat trong so

Voi gradient `dW` tinh duoc tu (mini-)batch hien tai, cau hoi la: **cap nhat `W` nhu the nao?**

### 2.1. Gradient Descent (GD) thuan

Don gian nhat - di chuyen `W` nguoc huong gradient mot luong co dinh `\alpha` (learning rate):

$$W := W - \alpha \, dW$$

Nhuoc diem: neu ham cost co dang "thung lung hep, dai" (common trong khong gian tham so nhieu
chieu), GD thuan de dao dong qua lai theo huong hep thay vi tien nhanh theo huong dai, lam cham
hoi tu.

### 2.2. Gradient Descent + Momentum

Y tuong: thay vi chi nhin gradient **hien tai**, tich luy mot "trung binh dong" (exponentially
weighted moving average) cua gradient qua cac buoc truoc, giong nhu mot **vien bi co da** lan
xuong doc - da tich luy giup no vuot qua cac dao dong nho va tang toc theo huong nhat quan:

$$v_{dW} := \beta \, v_{dW} + (1-\beta)\, dW \qquad\qquad W := W - \alpha \, v_{dW}$$

`\beta` (thuong `0.9`) quyet dinh "trong luong" cua qua khu - `\beta` cang gan 1, da cang "nang",
cang lam muot dao dong nhung cung phan ung cham hon voi thay doi moi.

### 2.3. Adam (Adaptive Moment Estimation)

Ket hop **Momentum** (trung binh dong bac 1 cua gradient, `v`) **va** y tuong cua RMSProp - trung
binh dong bac 2 cua **binh phuong** gradient (`s`), dung de **tu dieu chinh learning rate rieng cho
tung tham so**: tham so co gradient lich su lon se duoc "giam toc" nhieu hon, tham so co gradient
nho se duoc "tang toc":

$$v_{dW} := \beta_1 v_{dW} + (1-\beta_1) dW \qquad\qquad s_{dW} := \beta_2 s_{dW} + (1-\beta_2) dW^2$$
$$W := W - \alpha \, \frac{v_{dW}}{\sqrt{s_{dW}} + \varepsilon}$$

(Adam con co buoc "bias correction" cho `v`, `s` o cac iteration dau, khi trung binh dong con thien
lech ve 0.) Adam thuong hoi tu nhanh hon GD/Momentum va **it can dieu chinh learning rate thu cong**
hon, nen la lua chon mac dinh pho bien trong thuc te.

## 3. Du lieu & kien truc

Ta huan luyen mot mang `[8, 5, 2, 1]` (2 hidden layer, khoi tao He) tren bo du lieu y te that
**Pima Indians Diabetes** (768 benh nhan, 8 dac trung lam sang, du doan tieu duong), voi 3 optimizer
khac nhau de so sanh toc do hoi tu.

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

from keras_utils import load_dataset, build_model, compile_and_train, evaluate_classification

tf.config.experimental.enable_op_determinism()  # dam bao ket qua giong het nhau giua cac lan chay

train_X, train_Y, test_X, test_Y = load_dataset()
print("train_X:", train_X.shape, " train_Y:", train_Y.shape)

## 4. Chon optimizer - Bai tap

`tf.keras.optimizers` da cung cap san ca 3 thuat toan o muc 2:

- `keras.optimizers.SGD(learning_rate=...)` - GD thuan.
- `keras.optimizers.SGD(learning_rate=..., momentum=0.9)` - them tham so `momentum` la co Momentum.
- `keras.optimizers.Adam(learning_rate=..., beta_1=0.9, beta_2=0.999)` - Adam day du (bias
  correction da duoc cai san ben trong).

### Bai tap - `get_optimizer`

In [ ]:
def get_optimizer(name, learning_rate=0.0007):
    '''
    Tra ve mot optimizer Keras tuong ung: "gd" | "momentum" | "adam".

    Returns: mot tf.keras.optimizers.Optimizer
    '''
    # (~6 dong) dung keras.optimizers.SGD (co/khong momentum) va keras.optimizers.Adam
    # if name == "gd":
    #     ...
    # elif name == "momentum":
    #     ...
    # elif name == "adam":
    #     ...
    # YOUR CODE STARTS HERE
    pass
    # YOUR CODE ENDS HERE

In [ ]:
#@title Answer { display-mode: "form" }
def get_optimizer(name, learning_rate=0.0007):
    '''
    Tra ve mot optimizer Keras tuong ung: "gd" | "momentum" | "adam".

    Returns: mot tf.keras.optimizers.Optimizer
    '''
    # (~6 dong) dung keras.optimizers.SGD (co/khong momentum) va keras.optimizers.Adam
    # YOUR CODE STARTS HERE
    if name == "gd":
        return keras.optimizers.SGD(learning_rate=learning_rate)
    elif name == "momentum":
        return keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    elif name == "adam":
        return keras.optimizers.Adam(learning_rate=learning_rate, beta_1=0.9, beta_2=0.999)
    else:
        raise ValueError(f"Unknown optimizer name: {name}")
    # YOUR CODE ENDS HERE

## 5. Huan luyen & so sanh 3 optimizer

Dung cung `learning_rate=0.0007` va `batch_size=64` (mini-batch gradient descent, muc 1) cho ca 3
optimizer, de moi chenh lech ket qua chi den tu ban than thuat toan cap nhat.

In [ ]:
EPOCHS = 200
histories = {}
for name in ["gd", "momentum", "adam"]:
    tf.random.set_seed(3)
    model = build_model(train_X.shape[0])
    optimizer = get_optimizer(name, learning_rate=0.0007)
    history = compile_and_train(model, train_X, train_Y, optimizer, epochs=EPOCHS, batch_size=64, verbose=0)
    histories[name] = {"model": model, "history": history}
    print(f"{name:10s} -> cost cuoi cung (train) = {history.history['loss'][-1]:.4f}")

In [ ]:
plt.figure(figsize=(7, 4.5))
for name in ["gd", "momentum", "adam"]:
    plt.plot(histories[name]["history"].history["loss"], label=name)
plt.xlabel("epoch")
plt.ylabel("cost")
plt.title("Cost qua qua trinh train - so sanh optimizer")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
for name in ["gd", "momentum", "adam"]:
    evaluate_classification(histories[name]["model"], test_X, test_Y, f'optimizer = "{name}" (test set)')

## 6. Ket qua & Phan tich

Sau 200 epoch (`learning_rate=0.0007`, `batch_size=64`), ket qua tren test set: **`gd`** 67.5%
accuracy (Recall 25.9%), **`momentum`** 72.1% (Recall 38.9%), **`adam`** cao nhat voi 74.7%
accuracy, Recall 46.3%, F1 0.562. Cost cuoi cung: gd 0.638, momentum 0.502, adam **0.450**.

Thu tu **Adam > Momentum > GD** ve ca cost lan Recall the hien dung ly thuyet o muc 2: Momentum
tich luy da theo huong gradient nhat quan nen nhanh hon GD thuan; Adam ket hop them dieu chinh
learning rate thich nghi theo tung tham so nen hoi tu nhanh nhat trong ca ba. Ca 3 optimizer deu
con cach xa diem hoi tu hoan toan sau chi 200 epoch - voi learning rate co dinh nho nhu the nay,
can nhieu epoch hon nua (hang nghin) de GD/Momentum bat kip Adam.

**Diem mau chot:** voi cung mot kien truc, cung mot du lieu, chi doi thuat toan cap nhat trong so
da tao ra chenh lech dang ke ve toc do hoi tu trong cung mot ngan sach epoch - day la ly do lua
chon optimizer (va tinh chinh learning rate cho no) la mot buoc quan trong khi huan luyen mang
no-ron trong thuc te, khong chi la chi tiet ky thuat phu.

## 7. Bai tap mo rong

1. **Tang so epoch:** thu `epochs=2000` hoac `5000` cho `gd`/`momentum` - chung co dan bat kip
   `adam` khong? Dieu nay cho thay dieu gi ve moi lien he giua "toc do hoi tu" va "chat luong cuoi
   cung" cua mot optimizer?
2. **Tinh chinh learning rate rieng:** dung `get_optimizer(name, learning_rate=...)` de thu cac
   learning rate khac nhau cho tung optimizer (vd Adam thuong chiu duoc learning rate cao hon GD
   thuan) - tim learning rate giup moi optimizer hoi tu tot nhat trong 200 epoch.
3. **Hoc rate decay:** ket hop `keras.optimizers.schedules.ExponentialDecay` lam `learning_rate`
   truyen vao optimizer, de learning rate tu giam dan qua cac epoch - so sanh voi learning rate
   co dinh.
4. **Doi batch_size:** thu `batch_size=16` va `batch_size=256` cho `adam` - quan sat anh huong den
   do "muot" cua duong cost va toc do hoi tu (batch nho hon -> cap nhat nhieu hon/epoch nhung moi
   cap nhat "on ao" hon).

## 8. Tai lieu tham khao

| Chu de | Lien ket |
|---|---|
| Kingma & Ba, 2015 - Adam: A Method for Stochastic Optimization | https://arxiv.org/abs/1412.6980 |
| `tf.keras.optimizers` API | https://www.tensorflow.org/api_docs/python/tf/keras/optimizers |
| CS231n - Optimization notes (SGD/Momentum/Adam so sanh truc quan) | https://cs231n.github.io/neural-networks-3/#update |
| Pima Indians Diabetes Dataset (nguon goc) | https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database |